In [ ]:
import pandas as pd
import numpy as np

#load the data
df = pd.read_csv('Data/processed_loan_data.csv')

#1. text to integers e.g '36 months' to 36
df['term'] = df['term'].apply(lambda x: int(x.split()[0]))

#2. employment map: we'll map these scales to a simple  numeric scale.
emp_map = {
    '< 1 year': 0, '1 year': 1, '2 years': 2, '3 years': 3, '4 years': 4,
    '5 years': 5, '6 years': 6, '7 years': 7, '8 years': 8, '9 years': 9,
    '10+ years': 10
}
df['emp_length'] = df['emp_length'].map(emp_map)

#assuming NaN means unemployed or retired.
df['emp_length'] = df['emp_length'].fillna(-1)

#3. Convert sub-grade to numerical values, A1, A2 ... G5 tp 1-35.
all_grades = sorted(df['sub_grade'].unique())
grade_map = {grade: i+1 for i, grade in enumerate(all_grades)}
df['sub_grade_num'] = df['sub_grade'].map(grade_map)

print("Transformations complete!")
print(df[['term', 'emp_length', 'sub_grade_num']].head())

df['issue_d'].head()
#proceed to turn these strings shown above into a number of months relative to a reference point.

# Convert to datetime objects
df['issue_d'] = pd.to_datetime(df['issue_d'], format='%b-%Y')

# Find the earliest date
earliest_date = df['issue_d'].min()

# Calculate months elapsed from the start of the dataset
df['months_since_start'] = ((df['issue_d'].dt.year - earliest_date.year) * 12 + 
                            (df['issue_d'].dt.month - earliest_date.month))

# Drop the original date column (the model can't use it)
df.drop('issue_d', axis=1, inplace=True)

print(df[['months_since_start']].head())

df.select_dtypes(include=['object']).columns

# Identify remaining text columns
cat_cols = df.select_dtypes(include=['object']).columns

# Convert to Dummy Variables (One-Hot Encoding)
# drop_first=True prevents the Dummy Variable Trap
df_final = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# Final check: are there ANY non-numeric columns left?
print(f"Remaining Text Columns: {len(df_final.select_dtypes(include=['object']).columns)}")
print(f"New Total Shape: {df_final.shape}")

# Check how many unique values are in each text column
# We are looking for 'Criminals' with > 20-30 unique values
print(df.select_dtypes(include=['object']).nunique().sort_values(ascending=False))

# Convert to datetime
df['earliest_cr_line'] = pd.to_datetime(df['earliest_cr_line'], format='%b-%Y')

# Calculate the length of credit history in months
# Drop columns that are too unique or redundant
cols_to_drop = ['emp_title', 'zip_code', 'earliest_cr_line', 'title', 'grade']
df_pruned = df.drop(columns=cols_to_drop)

# Now, run the One-Hot Encoding on the REMAINING text columns
df_final = pd.get_dummies(df_pruned, drop_first=True)

print(f"Final Shape: {df_final.shape}")

# Drop rows that contain any NaN values
df_final = df_final.dropna()

print(f"New shape after dropping NaNs: {df_final.shape}")

# Drop columns that happen AFTER the loan is issued
# These are 'snitches' that tell the model the outcome
future_cols = [col for col in df_final.columns if 'last_credit_pull' in col 
               or 'last_pymnt' in col 
               or 'debt_settlement' in col 
               or 'settlement' in col]

df_final = df_final.drop(columns=future_cols)

print(f"Remaining columns: {df_final.shape[1]}")

# Create a list of 'leaky' patterns
leakage_patterns = ['loan_status', 'pymnt', 'out_prncp', 'funded_amnt', 'recoveries']

# Drop any column that matches these patterns
to_drop = [col for col in df_final.columns if any(p in col for p in leakage_patterns)]
df_honest = df_final.drop(columns=to_drop)

print(f"Purged {len(to_drop)} leaky columns.")
print(f"Final feature count: {df_honest.shape[1] - 1}") # -1 for target

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Define X (Features) and y (Target)
X = df_honest.drop('target_dummy', axis=1)
y = df_honest['target_dummy']

# 2. Split into 70% Training and 30% Testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 3. Scale the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training shape: {X_train_scaled.shape}")
print(f"Testing shape: {X_test_scaled.shape}")

#1. LOGISTIC REGRESSION.
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# 1. Initialize the Model
# We use 'max_iter=1000' to give the math enough time to converge
log_model = LogisticRegression(max_iter=1000)

# 2. Train (Fit) the model
log_model.fit(X_train_scaled, y_train)

# 3. Predict on the test set
predictions = log_model.predict(X_test_scaled)

# 4. Evaluate the performance
print("Confusion Matrix:")
print(confusion_matrix(y_test, predictions))
print("\nClassification Report:")
print(classification_report(y_test, predictions))

# Find the column that is 'snitching'
correlations = df_final.corr()['target_dummy'].sort_values(ascending=False)
print(correlations.head(10))

# 1. Initialize the Model with "Balanced" weights
log_model_balanced = LogisticRegression(max_iter=1000, class_weight='balanced')

# 2. Train
log_model_balanced.fit(X_train_scaled, y_train)

# 3. Predict
predictions_balanced = log_model_balanced.predict(X_test_scaled)

# 4. Evaluate
print("Balanced Confusion Matrix:")
print(confusion_matrix(y_test, predictions_balanced))
print("\nBalanced Classification Report:")
print(classification_report(y_test, predictions_balanced))

import matplotlib.pyplot as plt

# Get the coefficients (the 'weight' of each feature)
feature_importance = pd.DataFrame({'Feature': X.columns, 'Importance': log_model_balanced.coef_[0]})
feature_importance = feature_importance.sort_values(by='Importance', ascending=False)

# Show the top 10 things that INCREASE default risk
print("Top 10 Risk Factors:")
print(feature_importance.head(10))

# Show the top 10 things that DECREASE default risk
print("\nTop 10 Safety Factors:")
print(feature_importance.tail(10))

import joblib

# Save the model
joblib.dump(log_model_balanced, 'lending_club_model.pkl')

# Save the scaler (Crucial! You must scale new data the exact same way)
joblib.dump(scaler, 'lending_scaler.pkl')

print("Files saved: lending_club_model.pkl, lending_scaler.pkl")